# Human in the Loop

`HumanInTheLoopMiddleware` is a middleware that allows you to interrupt before tool calls.

```python
# Before executing tool it will interrupt based on the below configuration
HumanInTheLoopMiddleware(
    interrupt_on={
        "tool_name": True,  # All decisions (approve, edit, reject) allowed
        "tool_name": {"allowed_decisions": ["approve", "reject"]},  # No editing allowed
        "tool_name": False,  # No approval required
    },
    description_prefix="Tool execution pending approval",
)
```

Check for more details: https://docs.langchain.com/oss/python/langchain/human-in-the-loop


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command 

from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
from pprint import pprint


llm_model = ChatGroq(model_name="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"))

file_str = """FILE:::"""

@tool()
def write_file(content:str) -> bool:
    """ the tool for write a file, it will just override the content in a file
        args: content: str
        return True if return is done otherwise false
    """
    print("writting a file")
    global file_str
    file_str = content;
    return True

@tool()
def read_file() -> str:
    """ the tool for read a file, it will return the file content
        args: no args
        return content of the file as str
    """
    print("reading file")
    global file_str
    return file_str

agent = create_agent(
    model= llm_model,
    tools=[write_file, read_file],
    system_prompt=""" 
         You are a helpful assistant for writing/reading blogs.
        - When user asks to write a blog, call write_file(content=...).
        - When writing a file, do NOT output the file content, simply say "blog updated".
        - When reading a file, call read_file() and give the summarize the result content.
        """,
    middleware=[
        HumanInTheLoopMiddleware( 
            interrupt_on={
                "write_file": True,  # All decisions (approve, edit, reject) allowed
                "execute_sql": {"allowed_decisions": ["approve", "reject"]},  # No editing allowed
                "read_data": False, # no approval required
            },
            # Prefix for interrupt messages - combined with tool name and args to form the full message
            # e.g., "Tool execution pending approval: execute_sql with query='DELETE FROM...'"
            # Individual tools can override this by specifying a "description" in their interrupt config
            description_prefix="Tool execution pending approval",
        ),
    ],
    # Human-in-the-loop requires checkpointing to handle interrupts.
    # In production, use a persistent checkpointer like AsyncPostgresSaver.
    checkpointer=InMemorySaver(),  
)

config = {"configurable" : {"thread_id": "unique_id_123"}}

print("****Ask for blog write. Ex: Write a blog for virus infection in a file ***")

waiting_for_input = None
while True:
    input_msg = input( waiting_for_input if  waiting_for_input else "You")
    if input_msg in {"quit", "exit"}:
        break
    if waiting_for_input:
        result = agent.invoke(
            Command(
                resume={
                    "decisions": [
                        {
                            "type": input_msg 
                        }
                    ]
                }
            ),
            config = config
        )
        waiting_for_input = None
    else:
        result = agent.invoke({"messages": input_msg}, config = config)

    # interrupt occured
    interrupt_payload = result.get("__interrupt__")
    if interrupt_payload:
        waiting_for_input = "Are you want write now? approve/reject"
    pprint(result["messages"][-1])

# Custom Human in the Loop (Interrupts)

- Interrupts allow you to pause graph execution at specific points and wait for external input before continuing.
- Interrupt payloads surface as __interrupt__, so we can read it from graph_response."__interrupt__"
- interrupt allow any JSON-serializable value 
- re-invoking graph with "command with resume value", will resume the graph

** what will happen when interrupt() calls **
- Graph execution gets suspended
- State is saved using checkpointer
- Value is returned caller to __interrupt__
- Graph waits indefinitely untill you re-invoke with command

** resume the interrupt **
- resume can be done by calling Command(resume=...)
- The value passed to Command(resume=...) becomes the return value of the interrupt call
- while resuming the node restarts will start from begining, mean node contain interrupt() call from line 1.
- inerrupt()  will return the resume value (any JSON-serializable ) passed during Command(resume=...)

** note "" 
- the graph internally throws an exception while calling interrupt(), so we shouldn't wrap the interrupt() inside the try/catch block
- while resuming the node restarts will start from begining, mean node contain interrupt() call from line 1. 
- the above point is important so we shouldn't  do any db operation inside the interrupt node



In [ ]:
import string
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import ToolRuntime, tool
from langgraph.types import Command,interrupt

from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
from pprint import pprint


llm_model = ChatGroq(model_name="openai/gpt-oss-120b", api_key=os.getenv("GROQ_API_KEY"))

file_str = """FILE:::"""

@tool()
def ask_user(question:str)->str:
    """ to get the input from the user
        args: question: str
        return userinput str
    """
    print("invoking interrupt") # this will be called on re-invoking
    userinput = interrupt({
        "question": question
    
    })
    print("got user interrupt and answer")
    print(f"Got user input: {userinput}")
    return userinput


@tool()
def write_file(content:str) -> bool:
    """ the tool for write a file, it will just override the content in a file
        args: content: str
        return True if return is done otherwise false
    """
    print("writting a file")
    global file_str
    file_str = content;
    return True

@tool()
def read_file() -> str:
    """ the tool for read a file, it will return the file content
        args: no args
        return content of the file as str
    """
    print("reading file")
    global file_str
    return file_str


agent = create_agent(
    model=llm_model,
    tools = [ask_user, write_file, read_file],
    checkpointer=InMemorySaver(),
    system_prompt=""" 
          You are a helpful assistant for writing/reading blogs.
        - You must clarify user questions with help of ask_user tool and complete the blogs
        - When user asks to write a blog, call write_file(content=...).
        - When writing a file, do NOT output the file content, simply say "blog updated".
        - When reading a file, call read_file() and give the summarize the result content.
    """
)


config = {"configurable" : {"thread_id": "unique_id_tool"}}


waiting_for_input = None
while True:
    input_msg = input( waiting_for_input if  waiting_for_input else "You")
    if input_msg in {"quit", "exit"}:
        break
    if waiting_for_input:
        result = agent.invoke(
            Command(
                resume=input_msg
            ),
            config = config
        )
        waiting_for_input = None
    else:
        result = agent.invoke({"messages": input_msg}, config = config)

    # interrupt occured
    interrupt_payload = result.get("__interrupt__")
    if interrupt_payload:
        print(interrupt_payload)
        waiting_for_input = interrupt_payload[0].value["question"]
    pprint(result["messages"][-1])
    

/Users/sureshkumars/Documents/AI_learnings/ai-learning-rag-chain/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


invoking interrupt
[Interrupt(value={'question': 'Sure! Could you please provide some details for the blog? For example, the topic, desired length (e.g., word count or number of sections), tone/style (e.g., formal, casual, persuasive), and any specific points you want covered.'}, id='9674c3a00f9b84da8a899c1791184d59')]
AIMessage(content='', additional_kwargs={'reasoning_content': 'The user says "write a blog". We need to ask for clarification: what topic, length, style? Use ask_user tool.', 'tool_calls': [{'id': 'fc_faad17f3-ec33-4d26-8e3b-4465a0295b9f', 'function': {'arguments': '{"question":"Sure! Could you please provide some details for the blog? For example, the topic, desired length (e.g., word count or number of sections), tone/style (e.g., formal, casual, persuasive), and any specific points you want covered."}', 'name': 'ask_user'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 104, 'prompt_tokens': 316, 'total_tokens': 420, 'completion_time': 